# Arranque operativo en Google Colab RTX PRO 6000 Blackwell (`sm_120`)

Cuaderno local de ejecución por pasos ordenados sobre el kit de `mars-titan`.
- **Contrato numérico:** FP32 IEEE estricto (sin TF32 ni precisión mixta, `TRITON_F32_DEFAULT=ieee`) y algoritmos deterministas.
- **Datos:** restaurados en `/content` desde el paquete que ya está en Google Drive, con la huella SHA-256 de cada volumen. Desde aquí solo sale el estado de la campaña hacia su espejo.
- **Seguridad temporal:** solo orígenes hasta 2023; el año de la evaluación final sigue sellado.
- **Relevos:** una sesión dura menos de 24 h. El paso 6 para la cola en su barrera, copia el estado al espejo y solo entonces deja soltar la máquina. La sesión siguiente repite los pasos 0 a 4 y la cola continúa donde quedó.


In [ ]:
# Paso 0: variables del entorno y montaje de Drive (solo en Colab)
import os

os.environ['MARS_TITAN_SITE'] = 'google-colab-rtx-pro-6000'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['TRITON_F32_DEFAULT'] = 'ieee'

# Ajustar antes de ejecutar (rutas dentro de Drive montado en /content/drive):
os.environ['P'] = '/content/drive/MyDrive/mars-titan-paquete'
os.environ['B'] = '/content/drive/MyDrive/mars-titan-paquete/mars-titan-colab.bundle'
os.environ['RAMA_KIT'] = 'dgx-trainning'  # la rama de las herramientas, no la del modelo
# COMMIT_HEX falla en cerrado mientras no sea el commit autorizado, con sus 40 hex
os.environ['COMMIT_HEX'] = 'REEMPLAZAR_CON_COMMIT_40_HEX'
os.environ['T'] = '/content/mars-titan'
os.environ['K'] = '/content/mars-titan/kit/scripts'
os.environ['ADDON'] = '/content/mars-titan/kit/scripts/colab_rtx6000'
os.environ['ESTADO'] = '/content/mars-titan/estado/v11'
os.environ['MIRROR_DRIVE'] = '/content/drive/MyDrive/mars-titan-estado-colab/v11'
os.environ['DESTINO_DATOS'] = '/home/gonzalo'
vistas = '/home/gonzalo/.local/state/mars-titan/campaign-a-v2-views-v31-20261013/views'
os.environ['VISTAS'] = vistas + '/US'  # '/US+CN' para la campaña conjunta
os.environ['FACTOR_CN_ZIP'] = ''  # con vistas US+CN: zip del factor de mercado de China
os.environ['DECLARACION'] = 'configs/experiments/us-campaign-v11-screen1.json'
os.environ['RECETA'] = 'configs/titans/dgx-us-core-v2.json'  # la de las entradas, como en la DGX
os.environ['ORIGIN_YEAR'] = '2013'
os.environ['SEMILLAS'] = '42 43 44'
os.environ['NIVELES'] = 'core'
os.environ['TRABAJADORES'] = '4'
os.environ['NOTA_PERMISO'] = 'Arranque en Colab RTX PRO 6000'
os.environ['COLAB_CU_PANEL'] = '7.8 CU/h'  # sustituir por la lectura real del panel de recursos
os.environ['SESSION_ID'] = 'relay-01'
os.environ['ESPERA_PARADA_S'] = '1800'  # margen para que la cola llegue a su barrera

try:
    from google.colab import drive  # type: ignore[import-not-found]
    drive.mount('/content/drive')
except ImportError:
    print('Fuera de Colab: se omite drive.mount')


## Paso 1 — Arranque repetible del equipo (`colab_bootstrap.sh` y `colab_site.py`)
Clona el kit desde el bundle en Drive, instala las herramientas, crea el entorno bloqueado, restaura los datos con su huella (y el factor de mercado de China si se da su zip), compila el enlace episódico nativo y anota la ficha del equipo.


In [ ]:
%%bash
set -euo pipefail
[ -d "$T/kit" ] || git clone --quiet --branch "$RAMA_KIT" "$B" "$T/kit"
bash "$K/colab_bootstrap.sh" sistema
bash "$K/colab_bootstrap.sh" codigo --bundle "$B" --commit "$COMMIT_HEX" --trabajo "$T"
bash "$K/colab_bootstrap.sh" entorno --trabajo "$T"
datos=(--paquete "$P" --trabajo "$T" --destino "$DESTINO_DATOS")
[ -z "${FACTOR_CN_ZIP:-}" ] || datos+=(--factor-cn "$FACTOR_CN_ZIP")
bash "$K/colab_bootstrap.sh" datos "${datos[@]}"
bash "$K/colab_bootstrap.sh" nativo --trabajo "$T"
bash "$K/colab_bootstrap.sh" comprobar --trabajo "$T"
"$T/repo/.venv/bin/python" "$K/colab_site.py" --output "$T/equipo.json" --disk "$T"


## Paso 2 — FP32 IEEE estricto y la escritura fusionada en `sm_120`
Comprueba, con el intérprete del entorno de la campaña, que el contrato numérico se puede imponer, y pasa el nivel 0 (bit a bit) y el nivel 1 (frente a FP64) del kernel antes de entrenar.


In [ ]:
%%bash
set -euo pipefail
export CUBLAS_WORKSPACE_CONFIG=:4096:8 TRITON_F32_DEFAULT=ieee
# Con el intérprete del entorno de la campaña, no con el del cuaderno.
"$T/repo/.venv/bin/python" - <<'PY'
import torch
import triton

torch.use_deterministic_algorithms(True, warn_only=False)
torch.set_float32_matmul_precision('highest')
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
assert triton.knobs.language.fp32_default == 'ieee', triton.knobs.language.fp32_default
assert not torch.backends.cuda.matmul.allow_tf32 and not torch.backends.cudnn.allow_tf32
print('FP32 IEEE estricto y determinismo: torch', torch.__version__, '| triton', triton.__version__)
PY


In [ ]:
%%bash
set -euo pipefail
export CUBLAS_WORKSPACE_CONFIG=:4096:8 TRITON_F32_DEFAULT=ieee
"$T/repo/.venv/bin/python" "$ADDON/kernel_sm120_plan.py" --mode cpu-verify
"$T/repo/.venv/bin/python" "$ADDON/kernel_sm120_plan.py" --mode ptx-audit
"$T/repo/.venv/bin/python" "$ADDON/kernel_sm120_plan.py" --mode gpu-sweep \
  --output "$T/sm120_kernel_report.json"


## Paso 3 — CUDA MPS y sonda de trabajadores (1, 2, 4 y 6)
Si MPS arranca y responde se deja activo y la cola lo usará; si no, la cola corre con procesos independientes. La sonda mide memoria y pasos por minuto para elegir `TRABAJADORES`.


In [ ]:
%%bash
set -euo pipefail
export CUBLAS_WORKSPACE_CONFIG=:4096:8 TRITON_F32_DEFAULT=ieee
"$T/repo/.venv/bin/python" "$ADDON/mps_probe.py" --probe-live --keep-running \
  --workers "$TRABAJADORES" --output "$T/mps_report.json"
# El tope de memoria por cliente no se pasa a la cola: la campaña acota la suya, como en la DGX.
"$T/repo/.venv/bin/python" - "$T/mps_report.json" "$T/mps_entorno.sh" <<'PY'
import json
import shlex
import sys

report = json.load(open(sys.argv[1], encoding='utf-8'))
names = ('CUDA_MPS_PIPE_DIRECTORY', 'CUDA_MPS_LOG_DIRECTORY') if report['mps_operational'] else ()
lines = [f"export {name}={shlex.quote(report['active_environment'][name])}" for name in names]
open(sys.argv[2], 'w', encoding='utf-8').write("".join(line + "\n" for line in lines))
print('MPS activo para la cola' if lines else 'Sin MPS: procesos independientes',
      '| trabajadores aconsejados:', report['recommended_workers'])
PY
"$T/repo/.venv/bin/python" "$ADDON/concurrency_probe.py" plan \
  --kit "$K" --repo "$T/repo" --state-root "$T/estado/sonda" \
  --views "$VISTAS" --declaration "$DECLARACION" \
  --origin "$ORIGIN_YEAR" --site "$MARS_TITAN_SITE"
# Tras la sonda en vivo (1, 2, 4 y 6), el análisis aconseja TRABAJADORES:
# "$T/repo/.venv/bin/python" "$ADDON/concurrency_probe.py" analyze \
#   --probe-root "$T/estado/sonda" --site-json "$T/equipo.json" --output "$T/sonda_resumen.json"


## Paso 4 — Reanudación desde el espejo, preparación del origen y cola
Recupera el estado anterior si existe (`colab_sync.py pull`), prepara cohortes, entradas y permiso (`colab_origin.sh`) y lanza la cola en `tmux`.


In [ ]:
%%bash
set -euo pipefail
export CUBLAS_WORKSPACE_CONFIG=:4096:8 TRITON_F32_DEFAULT=ieee
if [ -d "$MIRROR_DRIVE" ]; then
  "$T/repo/.venv/bin/python" "$K/colab_sync.py" pull --state "$ESTADO" --mirror "$MIRROR_DRIVE"
fi
origen=(--repo "$T/repo" --estado "$ESTADO" --origen "$ORIGIN_YEAR" --vistas "$VISTAS"
        --declaracion "$DECLARACION")
bash "$K/colab_origin.sh" cohortes "${origen[@]}"
bash "$K/colab_origin.sh" entradas "${origen[@]}" --receta "$RECETA"
bash "$K/colab_origin.sh" permiso "${origen[@]}" --nota "$NOTA_PERMISO"


In [ ]:
%%bash
set -euo pipefail
origen=(--repo "$T/repo" --estado "$ESTADO" --origen "$ORIGIN_YEAR" --vistas "$VISTAS"
        --declaracion "$DECLARACION")
# La cola corre en tmux desde un guion que lleva escrito su entorno: no depende del que tuviera
# un servidor tmux anterior, cada argumento conserva sus espacios y anota su sesión para la parada.
{
  echo '#!/usr/bin/env bash'
  echo 'set -euo pipefail'
  printf 'export %q\n' "MARS_TITAN_SITE=$MARS_TITAN_SITE" CUBLAS_WORKSPACE_CONFIG=:4096:8 \
    TRITON_F32_DEFAULT=ieee
  [ ! -s "$T/mps_entorno.sh" ] || cat "$T/mps_entorno.sh"
  printf 'ps -o sid= -p $$ | tr -d " " > %q\n' "$T/campana.sid"
  printf 'exec bash'
  printf ' %q' "$K/colab_origin.sh" lanza "${origen[@]}" --semillas "$SEMILLAS" \
    --niveles "$NIVELES" --trabajadores "$TRABAJADORES"
  printf '\n'
} > "$T/lanza_campana.sh"
tmux new-session -d -s campana \
  "bash $(printf '%q' "$T/lanza_campana.sh") 2>&1 | tee -a $(printf '%q' "$T/campana.log")"
bash "$K/colab_origin.sh" estado "${origen[@]}"
# Puerta de paridad con la DGX, cuando los dos estados tienen el mismo trabajo:
# "$T/repo/.venv/bin/python" "$K/colab_parity.py" --a <estado de la DGX> --b "$ESTADO" \
#   --model titans_mac --origin 2013 --seed 42 --output "$T/paridad.json"


## Paso 5 — Copia periódica al espejo y guarda de unidades de cómputo
Copia el estado a Drive y decide si se sigue con el siguiente trabajo u origen o se para antes del techo de 22,5 h o de la reserva de unidades. Las cifras de pasos son las medidas en esta sesión.


In [ ]:
%%bash
set -euo pipefail
"$T/repo/.venv/bin/python" "$K/colab_sync.py" push --state "$ESTADO" --mirror "$MIRROR_DRIVE"
"$T/repo/.venv/bin/python" "$ADDON/cu_budget_guard.py" decide \
  --panel-text "$COLAB_CU_PANEL" \
  --budget-cu 1000 \
  --session-elapsed-hours 2.0 \
  --remaining-steps 18000 \
  --steps-per-min 320.0 \
  --next-origin "$ORIGIN_YEAR" \
  --ledger "$MIRROR_DRIVE/cu_ledger.json" \
  --session-id "$SESSION_ID" \
  --state-dir "$ESTADO" \
  --mirror-dir "$MIRROR_DRIVE" \
  --kit-dir "$K"


## Paso 6 — Cierre seguro antes de 22,5 h o de agotar la reserva de unidades
Pide la parada al proceso de la cola, espera a que llegue a su barrera, copia el estado al espejo y detiene MPS. Si la cola no para, no copia ni deja soltar la máquina.


In [ ]:
%%bash
set -euo pipefail
# La parada la recibe el proceso de la cola, que para en su siguiente barrera. Se busca solo en la
# sesión que anotó el lanzador, nunca por nombre en toda la máquina.
rm -f "$T/final_sync.json"
patron='mars_titan.training.comparison_campaig[n] run'
viva() {
  local codigo=0
  pgrep -s "$sid" -f "$patron" >/dev/null || codigo=$?
  [ "$codigo" -le 1 ] || { echo "pgrep falló con código $codigo" >&2; exit 1; }
  [ "$codigo" = 0 ]
}
if [ -s "$T/campana.sid" ]; then
  sid="$(cat "$T/campana.sid")"
  [[ "$sid" =~ ^[1-9][0-9]*$ ]] || { echo "Sesión de la cola no válida: $sid" >&2; exit 1; }
  pkill -TERM -s "$sid" -f "$patron" || true
  for _ in $(seq 1 "${ESPERA_PARADA_S:-1800}"); do
    viva || break
    sleep 1
  done
  if viva; then
    echo "La cola sigue viva tras la espera: no se copia ni se suelta la máquina" >&2
    exit 1
  fi
  rm -f "$T/campana.sid"
fi
"$T/repo/.venv/bin/python" "$K/colab_sync.py" push --state "$ESTADO" --mirror "$MIRROR_DRIVE" \
  > "$T/final_sync.json.tmp"
mv "$T/final_sync.json.tmp" "$T/final_sync.json"
cat "$T/final_sync.json"
if [ -s "$T/mps_entorno.sh" ] && command -v nvidia-cuda-mps-control >/dev/null 2>&1; then
  ( . "$T/mps_entorno.sh"; echo quit | nvidia-cuda-mps-control ) || true
fi


In [ ]:
# La máquina se suelta solo si la copia final no dejó nada retenido ni pendiente de reintento
import json
import os
from pathlib import Path

AUTO_UNASSIGN = False  # Cambiar a True solo cuando se quiera liberar la máquina al acabar
sync_file = Path(os.environ.get('T', '/content/mars-titan')) / 'final_sync.json'
if AUTO_UNASSIGN and sync_file.is_file():
    report = json.loads(sync_file.read_text(encoding='utf-8'))
    if not report.get('held') and not report.get('retried_later'):
        from google.colab import runtime  # type: ignore[import-not-found]
        runtime.unassign()
